# 08 — Phase 1.A: hand-correct the desktop test split (Roboflow)

**Goal of this notebook (Phase 1.A.1 + 1.A.2):** turn the auto-labelled test split — which the headline desktop fine-tune was *evaluated against* — into a hand-corrected ground-truth split. Without this, the 0.7176 mAP@.5 in §4.1 measures *agreement with the auto-labeller*, not absolute correctness, and is not directly comparable to the source baseline's 0.4505 (real human GT). After this notebook the comparison is honest.

## What we do

1. **Bootstrap** the test split images + auto-labels onto a fresh Colab `/content/desktop_finetune/` (or use what's already there).
2. **Preview** the current auto-labels with Ultralytics overlays so you can see what needs fixing (e.g. the Windows 11 Save button missed in the 5 May 2026 demo).
3. **Package** the test split into a single zip you upload to Roboflow.
4. *(You hand-correct in Roboflow's web UI — instructions in Section 3.)*
5. **Import** the corrected zip back into Drive, replacing the old test labels.
6. **Verify** (Phase 1.A.2): count per-class instances, sanity-check, write `tables/desktop_test_handcorrected.csv` for the report.

## Why Roboflow and not LabelImg / CVAT

- 7–8 images is too small to justify standing up CVAT.
- Roboflow free-tier accepts the zip directly, imports auto-labels as starting annotations (you only correct, you don't re-label from scratch), and exports YOLOv8 labels back as a single zip.
- LabelImg works fine but requires a local Python install and one-image-at-a-time. Roboflow's grid view is faster for 8 images.
- If you prefer LabelImg or CVAT: the zip in Section 2 is YOLO-format and works in any tool that imports YOLO.

## Roadmap pointer

Tracks Plan §L.1.A in `docs/VisClick_Detailed_Plan.md`. After this notebook, mark `1.A.1` and `1.A.2` complete and move on to `08_phase1B_ablations.ipynb`.

## 0 — Setup (mount Drive, restore test split if needed)

Idempotent: if `/content/desktop_finetune/{images,labels}/test/` already has the 7–8 test items (e.g. you just ran `06`), this cell is a no-op. Otherwise it extracts `<DRIVE>/data/desktop_finetune_bundles/test.tar.gz` (the persistent bundle written by `06.3`).

In [ ]:
import os, glob, tarfile, shutil, time, json
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive/visclick"
except ImportError:
    DRIVE = os.environ.get("VISCLICK_DRIVE", "./drive")
    Path(DRIVE).mkdir(parents=True, exist_ok=True)

ROOT      = "/content/desktop_finetune"
BUNDLES   = os.path.join(DRIVE, "data", "desktop_finetune_bundles")
TEST_IMGS = os.path.join(ROOT, "images", "test")
TEST_LBLS = os.path.join(ROOT, "labels", "test")
Path(TEST_IMGS).mkdir(parents=True, exist_ok=True)
Path(TEST_LBLS).mkdir(parents=True, exist_ok=True)

def _count(p, exts):
    return len([f for f in os.listdir(p) if f.lower().endswith(exts)]) if os.path.isdir(p) else 0

n_imgs = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
n_lbls = _count(TEST_LBLS, (".txt",))
if n_imgs >= 5 and n_lbls >= 5:
    print(f"[skip] test split already present: {n_imgs} images / {n_lbls} labels")
else:
    bundle = os.path.join(BUNDLES, "test.tar.gz")
    assert os.path.isfile(bundle), (
        f"missing {bundle}.\n"
        f"Run 06_finetune_desktop.ipynb up to 6.3 first to produce the test bundle."
    )
    print(f"[restore] extracting {bundle}")
    with tarfile.open(bundle, "r:gz") as tf:
        tf.extractall(ROOT)
    n_imgs = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
    n_lbls = _count(TEST_LBLS, (".txt",))
    print(f"[restore] now {n_imgs} images / {n_lbls} labels")

assert n_imgs == n_lbls, f"mismatch: {n_imgs} images vs {n_lbls} labels — fix before continuing"
print(f"REPORT step = SETUP | n_test = {n_imgs}")

## 1 — Preview current auto-labels (so you know what to fix)

Renders the 7–8 test images with their auto-label overlays. Look for:
- Save / OK / Apply buttons that the auto-labeller missed (the live demo's classic failure case).
- `text` boxes that should actually be `button`, `text_input`, or `menu`.
- Dropdown arrows / close-X / settings cogs not labelled at all (these are `icon`).
- Wrong class assignments overall.

These are the corrections you'll make in Roboflow.

In [ ]:
import matplotlib.pyplot as plt
import cv2

CLS = ["button", "text", "text_input", "icon", "menu", "checkbox"]
COLOURS = [(255,80,80),(255,200,0),(0,180,255),(40,200,40),(180,80,255),(255,160,0)]

def draw_yolo(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    if not os.path.isfile(lbl_path):
        return img, 0
    n = 0
    with open(lbl_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                continue
            c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            x1 = int((xc - w/2) * W); y1 = int((yc - h/2) * H)
            x2 = int((xc + w/2) * W); y2 = int((yc + h/2) * H)
            colour = COLOURS[c % len(COLOURS)]
            cv2.rectangle(img, (x1,y1), (x2,y2), colour, 3)
            cv2.putText(img, CLS[c], (x1, max(y1-5, 12)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, colour, 2)
            n += 1
    return img, n

imgs = sorted(glob.glob(os.path.join(TEST_IMGS, "*")))
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, ip in zip(axes.flat, imgs[:8]):
    stem = os.path.splitext(os.path.basename(ip))[0]
    lp = os.path.join(TEST_LBLS, stem + ".txt")
    drawn, nb = draw_yolo(ip, lp)
    ax.imshow(drawn); ax.axis("off")
    ax.set_title(f"{os.path.basename(ip)}  ({nb} boxes)", fontsize=10)
for ax in axes.flat[len(imgs):]:
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Above are the test images with their CURRENT (auto-labelled) annotations.")
print("In the next step we package these for Roboflow upload; you correct them there.")

## 2 — Package test split into a Roboflow-ready zip

Produces a single `desktop_test_for_handlabel.zip` containing:
- `images/` — the 7–8 test PNG/JPG files
- `labels/` — the 7–8 YOLO `.txt` files (auto-labels, to be corrected)
- `data.yaml` — Roboflow reads this to set up the 6-class schema, so when you upload, the classes are already `button / text / text_input / icon / menu / checkbox`.

The zip lands at `<DRIVE>/data/desktop_test_for_handlabel.zip` — open Drive in your browser, right-click → Download. **Section 3 below has the step-by-step Roboflow workflow.**

In [ ]:
import zipfile

DATA_YAML = (
    "# generated by 08_phase1A_handlabel.ipynb\n"
    "path: .\n"
    "train: images\n"
    "val: images\n"
    "test: images\n"
    "\n"
    "nc: 6\n"
    "names: ['button', 'text', 'text_input', 'icon', 'menu', 'checkbox']\n"
)

OUT_ZIP_DIR = os.path.join(DRIVE, "data")
Path(OUT_ZIP_DIR).mkdir(parents=True, exist_ok=True)
OUT_ZIP = os.path.join(OUT_ZIP_DIR, "desktop_test_for_handlabel.zip")

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for ip in sorted(glob.glob(os.path.join(TEST_IMGS, "*"))):
        zf.write(ip, arcname=os.path.join("images", os.path.basename(ip)))
    for lp in sorted(glob.glob(os.path.join(TEST_LBLS, "*.txt"))):
        zf.write(lp, arcname=os.path.join("labels", os.path.basename(lp)))
    zf.writestr("data.yaml", DATA_YAML)

size_mb = os.path.getsize(OUT_ZIP) / 1024 / 1024
print(f"REPORT step = PACKAGE | path = {OUT_ZIP} | size_mb = {size_mb:.2f}")
print()
print("Next step:")
print(f"  1. Open Drive in your browser at {OUT_ZIP}")
print(f"  2. Right-click → Download (zip is ~{size_mb:.1f} MB)")
print(f"  3. Follow Section 3 below to upload + correct in Roboflow.")

## 3 — Roboflow workflow (the manual part)

Time budget: **~30 min** for 7–8 images.

### 3.1 — One-time Roboflow account

1. Go to <https://roboflow.com/> → **Sign up** (free public-workspace tier is fine for 8 images).
2. Create a workspace named `visclick` (or anything).

### 3.2 — Create the project

1. **Create new project**.
2. **Project type:** *Object Detection*.
3. **Annotation group:** `ui_element`.
4. **License:** *Private* (don't publish your screenshots).
5. Project name: `visclick-desktop-test-handlabel`.

### 3.3 — Upload the zip

1. **Upload** → drag in the `desktop_test_for_handlabel.zip` you downloaded from Drive.
2. Roboflow auto-detects YOLO format from `data.yaml`. Confirm classes look right (`button / text / text_input / icon / menu / checkbox`).
3. **Save and Continue** → assign all 8 to *training* (Roboflow's split is irrelevant; we only care about exporting corrected labels).

### 3.4 — Hand-correct the annotations

For each of the 8 images, open it in Roboflow's annotation tool and:

**(a) Find missed clickables.** Compare against the live screenshot. The Save / OK / Apply buttons are the most common misses. Add a `button` box for each.

**(b) Fix wrong class labels.** Anything labelled `text` that's actually a button, dropdown, menu, or text-input — correct the class. Use the dropdown arrow shortcut.

**(c) Add icon-only clickables.** Close-X title-bar buttons, settings cogs, dropdown arrows (`▼` / `^v`), minimise / maximise. Class = `icon`.

**(d) Delete spurious boxes.** Anything the auto-labeller hallucinated.

**(e) Tighten loose boxes.** Drag corners so the box hugs the element. (Doesn't have to be pixel-perfect; ~5 px slack is fine.)

**Tip:** keyboard shortcuts speed this up — `B` to draw box, `D` to delete selected, `1`–`6` to set class, `Tab` next image.

### 3.5 — Export

1. **Versions** → **Generate New Version**.
2. Preprocessing: leave at *Auto-Orient* only. **No augmentation** (we don't want Roboflow to fabricate training images for us).
3. **Generate**.
4. **Export Dataset** → format **YOLOv8** → **Download zip to computer**.
5. Upload that zip back into your Drive at `<DRIVE>/data/desktop_test_handcorrected.zip`. (The next code cell ingests it.)

### 3.6 — Sanity check before continuing

Open the downloaded zip locally and confirm:
- `train/images/` has your 7–8 PNG/JPG files (Roboflow may put them all in `train/` regardless of original split).
- `train/labels/` has matching `.txt` files.
- `data.yaml` has `nc: 6` and the same class names in the same order.

If any class names are mis-ordered, Section 4 below remaps them. Don't manually edit the labels.

## 4 — Import corrected labels back into the test split

Set `ROBOFLOW_ZIP` to the path of the zip you downloaded from Roboflow and uploaded to Drive. Default path matches the suggestion in Section 3.5.

What this cell does:
1. Extract the Roboflow zip into a temp dir.
2. Read its `data.yaml` to determine how Roboflow ordered the classes.
3. Remap each label file's class indices into VisClick's canonical order (`button=0 / text=1 / text_input=2 / icon=3 / menu=4 / checkbox=5`).
4. Replace `<DRIVE>/data/desktop_finetune_bundles/test.tar.gz` with a corrected bundle (so future runs of `06`, `08_phase1B_*` use the corrected labels automatically).
5. Also write a backup `test_autolabels.tar.gz` of the original bundle so you can revert if needed.

In [ ]:
import yaml

ROBOFLOW_ZIP = os.path.join(DRIVE, "data", "desktop_test_handcorrected.zip")
assert os.path.isfile(ROBOFLOW_ZIP), (
    f"Missing {ROBOFLOW_ZIP}.\n"
    f"Upload the YOLOv8 zip you downloaded from Roboflow to that path on Drive."
)

TMP = "/content/_rf_tmp"
shutil.rmtree(TMP, ignore_errors=True); Path(TMP).mkdir(parents=True)
with zipfile.ZipFile(ROBOFLOW_ZIP, "r") as zf:
    zf.extractall(TMP)

yml_path = None
for root, _, files in os.walk(TMP):
    if "data.yaml" in files:
        yml_path = os.path.join(root, "data.yaml"); break
assert yml_path, "data.yaml missing from Roboflow zip"
with open(yml_path) as f:
    rf_yaml = yaml.safe_load(f)
rf_names = list(rf_yaml.get("names", []))
print(f"Roboflow class order: {rf_names}")

VC_NAMES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
rf_to_vc = {}
for i, n in enumerate(rf_names):
    if n in VC_NAMES:
        rf_to_vc[i] = VC_NAMES.index(n)
    else:
        print(f"  WARNING: Roboflow class {n!r} not in VisClick taxonomy; will drop those boxes")

lbl_dirs = []
for root, _, files in os.walk(TMP):
    if any(f.endswith(".txt") for f in files):
        if root.endswith("labels") or os.path.basename(os.path.dirname(root)) == "labels":
            lbl_dirs.append(root)
img_dirs = []
for root, _, files in os.walk(TMP):
    if any(f.lower().endswith((".png", ".jpg", ".jpeg")) for f in files):
        if root.endswith("images") or os.path.basename(os.path.dirname(root)) == "images":
            img_dirs.append(root)
print(f"label dirs found: {lbl_dirs}\nimage dirs found: {img_dirs}")

if os.path.isfile(os.path.join(BUNDLES, "test.tar.gz")):
    backup = os.path.join(BUNDLES, "test_autolabels.tar.gz")
    if not os.path.isfile(backup):
        shutil.copy2(os.path.join(BUNDLES, "test.tar.gz"), backup)
        print(f"[backup] saved auto-label bundle to {backup}")

shutil.rmtree(TEST_IMGS, ignore_errors=True); Path(TEST_IMGS).mkdir(parents=True)
shutil.rmtree(TEST_LBLS, ignore_errors=True); Path(TEST_LBLS).mkdir(parents=True)

for d in img_dirs:
    for fn in os.listdir(d):
        if fn.lower().endswith((".png", ".jpg", ".jpeg")):
            shutil.copy2(os.path.join(d, fn), os.path.join(TEST_IMGS, fn))
for d in lbl_dirs:
    for fn in os.listdir(d):
        if not fn.endswith(".txt"):
            continue
        out = os.path.join(TEST_LBLS, fn)
        kept = 0
        with open(os.path.join(d, fn)) as fin, open(out, "w") as fout:
            for line in fin:
                parts = line.split()
                if len(parts) != 5:
                    continue
                c = int(parts[0])
                if c not in rf_to_vc:
                    continue
                fout.write(" ".join([str(rf_to_vc[c])] + parts[1:]) + "\n")
                kept += 1
        print(f"  {fn}: kept {kept} boxes")

n_imgs_new = _count(TEST_IMGS, (".png", ".jpg", ".jpeg"))
n_lbls_new = _count(TEST_LBLS, (".txt",))
print(f"REPORT step = IMPORT | n_imgs = {n_imgs_new} | n_labels = {n_lbls_new}")

import tarfile as _tar
new_bundle = os.path.join(BUNDLES, "test.tar.gz")
with _tar.open(new_bundle, "w:gz") as tf:
    for sub in ("images/test", "labels/test"):
        full = os.path.join(ROOT, sub)
        for fn in sorted(os.listdir(full)):
            tf.add(os.path.join(full, fn), arcname=os.path.join(sub, fn))
print(f"REPORT step = REPACK | new test.tar.gz = {os.path.getsize(new_bundle)} bytes")

## 5 — Phase 1.A.2: verify and write report row

Counts per-class instances on the corrected test split, sanity-checks (no negative coords, no out-of-range classes), and writes `<DRIVE>/reports/tables/desktop_test_handcorrected.csv` for direct paste into Report §4.6b.

In [ ]:
from collections import Counter
import csv

VC_NAMES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
counts = Counter()
errors = []
n_imgs = 0; n_boxes = 0

for ip in sorted(glob.glob(os.path.join(TEST_IMGS, "*"))):
    stem = os.path.splitext(os.path.basename(ip))[0]
    lp = os.path.join(TEST_LBLS, stem + ".txt")
    if not os.path.isfile(lp):
        errors.append(f"missing label for {os.path.basename(ip)}"); continue
    n_imgs += 1
    with open(lp) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                errors.append(f"{stem}: malformed line: {line.strip()!r}"); continue
            c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            if not (0 <= c < 6):
                errors.append(f"{stem}: class out of range: {c}"); continue
            for v, n in zip([xc, yc, w, h], ["xc", "yc", "w", "h"]):
                if not (0 <= v <= 1):
                    errors.append(f"{stem}: {n}={v} not in [0,1]")
            counts[VC_NAMES[c]] += 1
            n_boxes += 1

print(f"\nREPORT §4.6b — hand-corrected test split:")
print(f"  images : {n_imgs}")
print(f"  boxes  : {n_boxes}")
for cls in VC_NAMES:
    print(f"    {cls:11s} : {counts.get(cls, 0)}")
if errors:
    print("\nWARNINGS:")
    for e in errors[:10]:
        print("  " + e)
    if len(errors) > 10:
        print(f"  ... ({len(errors) - 10} more)")
else:
    print("\n✓ no warnings — labels look clean")

TABLES = os.path.join(DRIVE, "reports", "tables")
Path(TABLES).mkdir(parents=True, exist_ok=True)
csv_path = os.path.join(TABLES, "desktop_test_handcorrected.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["split", "images", "total_boxes"] + VC_NAMES + ["date"])
    w.writerow(["test", n_imgs, n_boxes] + [counts.get(c, 0) for c in VC_NAMES] +
                [time.strftime("%Y-%m-%d")])
print(f"\nREPORT step = VERIFY | path = {csv_path}")
print("Paste these counts into Report §4.6b. Then mark Plan 1.A.1 and 1.A.2 done and run 08_phase1B_ablations.ipynb.")

## 6 — What's next

- [x] **1.A.1** Hand-correct the 8 desktop test images in Roboflow → done above.
- [x] **1.A.2** Verify exports → CSV written to `<DRIVE>/reports/tables/desktop_test_handcorrected.csv`.
- [ ] **1.B.1–5** Run `08_phase1B_ablations.ipynb` (will be added next): trains M0 / M1 / M2, re-evaluates M3 (= current desktop FT) on this hand-corrected test set, dumps `tables/transfer_experiments.csv`.
- [ ] **1.C.1–3** On your Windows machine, run the three baseline scripts (`scripts/baseline_template.py` / `baseline_ocr_only.py` / `baseline_pywinauto.py`) — these will be added in a separate commit.

Update Plan §L.1 checkboxes and commit with a message like:

```
Phase 1.A.1: hand-corrected 8 desktop test images (n_boxes=NN)
Phase 1.A.2: verified counts, wrote tables/desktop_test_handcorrected.csv
```